# Evaluation Metric Formulas

This notebook documents every evaluation metric used in the **hybrid model training pipeline** (`hybrid_model_training_sequential.ipynb`) for the Automotive Spare Parts Demand Forecasting Decision Support System.

All metrics are computed over the **walk-forward validation** holdout predictions.

**Notation:**

| Symbol | Meaning |
| :---: | :--- |
| $y_t$ | Actual value at time $t$ |
| $\hat{y}_t$ | Predicted (forecasted) value at time $t$ |
| $n$ | Number of walk-forward test epochs |
| $\bar{y}$ | Mean of actual values: $\bar{y} = \frac{1}{n} \sum_{t=1}^{n} y_t$ |

---

## 1. Scale-Dependent Error Metrics

These metrics are expressed in the **same units** as the target variable (e.g., units sold, pesos). They are useful for comparing models on the **same series** but cannot be used to compare across series with different scales.

### 1.1 Mean Squared Error (MSE)

$$
\text{MSE} = \frac{1}{n} \sum_{t=1}^{n} (y_t - \hat{y}_t)^2
$$

- **Units**: Squared units of $y$ (e.g., units²)
- **Range**: $[0, \infty)$
- **Interpretation**: Average squared deviation between forecasts and actuals. Heavily **penalizes large errors** due to the squaring operation.
- **Limitation**: Not directly interpretable because the units are squared.

### 1.2 Root Mean Squared Error (RMSE)

$$
\text{RMSE} = \sqrt{\frac{1}{n} \sum_{t=1}^{n} (y_t - \hat{y}_t)^2} = \sqrt{\text{MSE}}
$$

- **Units**: Same as $y$ (e.g., units sold)
- **Range**: $[0, \infty)$
- **Interpretation**: Standard deviation of prediction errors. More interpretable than MSE because it shares the target's units. Still **penalizes large errors** more than MAE due to the squaring before averaging.
- **Usage**: Primary scale-dependent accuracy metric. Lower is better.

### 1.3 Mean Absolute Error (MAE)

$$
\text{MAE} = \frac{1}{n} \sum_{t=1}^{n} |y_t - \hat{y}_t|
$$

- **Units**: Same as $y$
- **Range**: $[0, \infty)$
- **Interpretation**: Average absolute deviation. More **robust to outliers** than RMSE because it does not square the errors.
- **Usage**: Preferred when all errors should be weighted equally regardless of magnitude.

---

## 2. Percentage Error Metrics

These metrics express error as a **percentage** of the actual value, enabling comparison across series with different scales.

### 2.1 Mean Absolute Percentage Error (MAPE)

$$
\text{MAPE} = \frac{100\%}{n} \sum_{t=1}^{n} \left| \frac{y_t - \hat{y}_t}{y_t} \right|
$$

> **Note:** Only computed over observations where $y_t \neq 0$.

- **Units**: Percentage (%)
- **Range**: $[0, \infty)$
- **Interpretation**: Average percentage error. Widely used in business and supply chain contexts.
- **Limitation**: **Undefined when $y_t = 0$** (division by zero). For intermittent demand with many zero-demand months, MAPE can be misleading or incalculable.
- **Threshold**: MAPE < 10% is generally considered excellent; < 20% is good.

### 2.2 Symmetric Mean Absolute Percentage Error (sMAPE)

$$
\text{sMAPE} = \frac{100\%}{n} \sum_{t=1}^{n} \frac{|y_t - \hat{y}_t|}{(|y_t| + |\hat{y}_t|) \,/\, 2}
$$

- **Units**: Percentage (%)
- **Range**: $[0, 200\%]$
- **Interpretation**: Symmetric alternative to MAPE. Treats over-prediction and under-prediction **equally** by normalizing against the average of actual and predicted values.
- **Advantage**: Better behaved when actuals are near zero, though still problematic when both $y_t = 0$ and $\hat{y}_t = 0$.
- **Reference**: Makridakis (1993).

### 2.3 Weighted Absolute Percentage Error (WAPE)

$$
\text{WAPE} = \frac{\sum_{t=1}^{n} |y_t - \hat{y}_t|}{\sum_{t=1}^{n} |y_t|} \times 100\%
$$

- **Units**: Percentage (%)
- **Range**: $[0, \infty)$
- **Interpretation**: Total absolute error as a percentage of total actual volume. Equivalent to $\frac{\text{MAE}}{\bar{|y|}} \times 100\%$. Naturally **weights high-volume periods more heavily** than low-volume periods.
- **Advantage**: Robust to zero values in individual periods (unlike MAPE). Standard metric in **retail and supply chain forecasting**.
- **Usage in pipeline**: Used as the **primary window selection criterion** in `select_best_training_window()`.

---

## 3. Scaled Error Metrics

### 3.1 Mean Absolute Scaled Error (MASE)

$$
\text{MASE} = \frac{\displaystyle \frac{1}{n} \sum_{t=1}^{n} |y_t - \hat{y}_t|}{\displaystyle \frac{1}{T-m} \sum_{t=m+1}^{T} |y_t^{\text{train}} - y_{t-m}^{\text{train}}|}
$$

Where:
- **Numerator**: MAE of the model's out-of-sample forecasts
- **Denominator**: MAE of the **in-sample naïve forecast** (random walk: $m = 1$, or seasonal naïve: $m = 12$ for monthly data)
- $T$ = number of training observations
- $m$ = seasonal period (default $m = 1$ for non-seasonal naïve)

| MASE Value | Interpretation |
| :---: | :--- |
| $\text{MASE} < 1$ | Model is **better** than the naïve baseline |
| $\text{MASE} = 1$ | Model performs **equal** to the naïve baseline |
| $\text{MASE} > 1$ | Model is **worse** than the naïve baseline |

- **Units**: Dimensionless (ratio)
- **Advantage**: Scale-independent, well-defined for zero values, and provides an intuitive baseline comparison.
- **Reference**: Hyndman & Koehler (2006). *"Another look at measures of forecast accuracy."* International Journal of Forecasting, 22(4), 679–688.

---

## 4. Goodness-of-Fit Metrics

### 4.1 R² (Coefficient of Determination)

$$
R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}} = 1 - \frac{\displaystyle \sum_{t=1}^{n} (y_t - \hat{y}_t)^2}{\displaystyle \sum_{t=1}^{n} (y_t - \bar{y})^2}
$$

Where:
- $SS_{\text{res}} = \sum (y_t - \hat{y}_t)^2$ — Residual sum of squares (unexplained variance)
- $SS_{\text{tot}} = \sum (y_t - \bar{y})^2$ — Total sum of squares (total variance)

| R² Value | Interpretation |
| :---: | :--- |
| $R^2 = 1$ | Perfect predictions |
| $R^2 > 0$ | Model explains variance **better** than predicting the mean |
| $R^2 = 0$ | Model is equivalent to always predicting $\bar{y}$ |
| $R^2 < 0$ | Model is **worse** than predicting the mean |

- **Units**: Dimensionless
- **Range**: $(-\infty, 1]$
- **Note**: Negative $R^2$ is common with small test sets (e.g., 12 walk-forward epochs) and highly volatile intermittent demand. It does not mean the model is useless — only that, on this specific test window, the mean was a better point predictor.

---

## 5. Log-Space Error Metrics

### 5.1 Log Loss (Mean Squared Logarithmic Error — MSLE)

$$
\text{LogLoss} = \frac{1}{n} \sum_{t=1}^{n} \left( \ln(1 + y_t) - \ln(1 + \hat{y}_t) \right)^2
$$

Equivalently:

$$
\text{LogLoss} = \frac{1}{n} \sum_{t=1}^{n} \left( \log_e \frac{1 + y_t}{1 + \hat{y}_t} \right)^2
$$

- **Units**: Dimensionless (log-squared scale)
- **Range**: $[0, \infty)$
- **Interpretation**: Penalizes **under-predictions** more heavily than over-predictions in percentage terms. The $\ln(1+x)$ transform compresses large values and stabilizes variance.
- **Note**: This is labeled "LogLoss" in the codebase but is technically **MSLE** (Mean Squared Logarithmic Error), not the binary cross-entropy log loss used in classification.

---

## 6. Statistical Significance Tests

### 6.1 Diebold-Mariano (DM) Test

Tests the null hypothesis that two competing forecasts have **equal predictive accuracy**.

**Step 1 — Compute the loss differential series:**

$$
d_t = e_{1,t}^2 - e_{2,t}^2
$$

Where $e_{1,t} = y_t - \hat{y}_{1,t}$ and $e_{2,t} = y_t - \hat{y}_{2,t}$ are the forecast errors of Model 1 and Model 2, respectively.

**Step 2 — Compute the DM test statistic:**

$$
\text{DM} = \frac{\bar{d}}{\sqrt{\hat{\sigma}_d^2 \,/\, n}}
$$

Where:
- $\bar{d} = \frac{1}{n} \sum_{t=1}^{n} d_t$ — mean loss differential
- $\hat{\sigma}_d^2 = \frac{1}{n-1} \sum_{t=1}^{n} (d_t - \bar{d})^2$ — sample variance of $d_t$

**Step 3 — Compute p-value from the Student's t-distribution:**

$$
p = 2 \cdot \left(1 - F_{t}\left(|\text{DM}|;\, \text{df} = n - 1\right)\right)
$$

Where $F_t$ is the CDF of the Student's t-distribution.

**Decision Rule:**

| Condition | Interpretation |
| :--- | :--- |
| $p < 0.05$ and $\bar{d} < 0$ | Model 1 is **significantly better** |
| $p < 0.05$ and $\bar{d} > 0$ | Model 2 is **significantly better** |
| $p \geq 0.05$ | **No significant difference** between models |

**Usage in pipeline:**
- *Hybrid vs. Standalone*: Tests if ARIMA+XGB is significantly better than ARIMA-only (and TSB+XGB vs. TSB-only)
- *Hybrid vs. Naïve*: Tests if the hybrid model is significantly better than the last-observed-value baseline

**Reference**: Diebold, F. X. & Mariano, R. S. (1995). *"Comparing predictive accuracy."* Journal of Business & Economic Statistics, 13(3), 253–263.

---

## 7. Hybrid Component Contribution Metrics

These metrics quantify the **value added by the XGBoost residual correction** component in the hybrid architecture. They compare the full hybrid model ($\hat{y}_{\text{hybrid}}$) against the standalone base model ($\hat{y}_{\text{standalone}}$).

### 7.1 MAE Improvement (%)

$$
\text{MAE Improvement} = \frac{\text{MAE}_{\text{standalone}} - \text{MAE}_{\text{hybrid}}}{\text{MAE}_{\text{standalone}}} \times 100\%
$$

- **Positive value**: XGB correction **reduces** MAE → the hybrid is better
- **Negative value**: XGB correction **increases** MAE → the base model alone is more accurate

### 7.2 RMSE Improvement (%)

$$
\text{RMSE Improvement} = \frac{\text{RMSE}_{\text{standalone}} - \text{RMSE}_{\text{hybrid}}}{\text{RMSE}_{\text{standalone}}} \times 100\%
$$

- Same interpretation as MAE Improvement but weighted toward larger errors.

### 7.3 Residual Variance Reduction (%)

$$
\text{Var. Reduction} = \frac{\text{Var}(\epsilon_{\text{standalone}}) - \text{Var}(\epsilon_{\text{hybrid}})}{\text{Var}(\epsilon_{\text{standalone}})} \times 100\%
$$

Where:
- $\epsilon_{\text{standalone}} = y_t - \hat{y}_{\text{standalone},t}$ — residuals from the base model
- $\epsilon_{\text{hybrid}} = y_t - \hat{y}_{\text{hybrid},t}$ — residuals from the hybrid model
- $\text{Var}(\epsilon) = \frac{1}{n} \sum_{t=1}^{n} (\epsilon_t - \bar{\epsilon})^2$

- **Positive value**: XGB reduces the **spread** of prediction errors
- **Interpretation**: Conceptually similar to the improvement in explained variance. A 25% reduction means the hybrid model's residuals have 25% less variance than the standalone model's residuals.

---

## 8. Standalone vs. Hybrid Forecast Definitions

The component contribution metrics compare two forecasts extracted from the same trained artifact:

### ARIMA + XGB Pipeline

$$
\hat{y}_{\text{standalone}} = \text{ARIMA}_{(p,d,q)}(h=1)
$$

$$
\hat{y}_{\text{hybrid}} = \text{ARIMA}_{(p,d,q)}(h=1) + \text{XGB}(\mathbf{x}_t)
$$

Where $\text{XGB}(\mathbf{x}_t)$ is the XGBoost correction trained on ARIMA's in-sample residuals.

### TSB + XGB Pipeline

$$
\hat{y}_{\text{standalone}} = \hat{p}_T \cdot \hat{z}_T
$$

$$
\hat{y}_{\text{hybrid}} = \hat{p}_T \cdot \hat{z}_T + \text{XGB}(\mathbf{x}_t)
$$

Where $\hat{p}_T$ is the TSB demand probability and $\hat{z}_T$ is the TSB demand size at the last training time step $T$.

### Naïve Forecast (Random Walk)

$$
\hat{y}_{\text{naïve},t} = y_{t-1}
$$

The simplest baseline: predict the last observed value.

---

## Summary Table

| # | Metric | Formula (compact) | Units | Best Value | Category |
| :---: | :--- | :--- | :---: | :---: | :--- |
| 1 | MSE | $\frac{1}{n}\sum(y - \hat{y})^2$ | units² | 0 | Scale-dependent |
| 2 | RMSE | $\sqrt{\text{MSE}}$ | units | 0 | Scale-dependent |
| 3 | MAE | $\frac{1}{n}\sum|y - \hat{y}|$ | units | 0 | Scale-dependent |
| 4 | MAPE | $\frac{100}{n}\sum\left|\frac{y-\hat{y}}{y}\right|$ | % | 0% | Percentage |
| 5 | sMAPE | $\frac{100}{n}\sum\frac{|y-\hat{y}|}{(|y|+|\hat{y}|)/2}$ | % | 0% | Percentage |
| 6 | WAPE | $\frac{\sum|y-\hat{y}|}{\sum|y|}\times100$ | % | 0% | Percentage |
| 7 | MASE | $\frac{\text{MAE}_{\text{model}}}{\text{MAE}_{\text{naïve}}}$ | — | < 1 | Scaled |
| 8 | R² | $1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}}$ | — | 1 | Goodness-of-fit |
| 9 | LogLoss | $\frac{1}{n}\sum(\ln(1+y)-\ln(1+\hat{y}))^2$ | — | 0 | Log-space |
| 10 | DM Test | $\frac{\bar{d}}{\sqrt{\hat{\sigma}_d^2/n}}$ | — | p < 0.05 | Significance |
| 11 | MAE Imp. % | $\frac{\text{MAE}_s - \text{MAE}_h}{\text{MAE}_s}\times100$ | % | > 0% | Contribution |
| 12 | RMSE Imp. % | $\frac{\text{RMSE}_s - \text{RMSE}_h}{\text{RMSE}_s}\times100$ | % | > 0% | Contribution |
| 13 | Var. Red. % | $\frac{\text{Var}(\epsilon_s) - \text{Var}(\epsilon_h)}{\text{Var}(\epsilon_s)}\times100$ | % | > 0% | Contribution |

---

## References

1. Hyndman, R. J. & Koehler, A. B. (2006). Another look at measures of forecast accuracy. *International Journal of Forecasting*, 22(4), 679–688.
2. Diebold, F. X. & Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3), 253–263.
3. Makridakis, S. (1993). Accuracy measures: theoretical and practical concerns. *International Journal of Forecasting*, 9(4), 527–529.
4. Syntetos, A. A. & Boylan, J. E. (2005). The accuracy of intermittent demand estimates. *International Journal of Forecasting*, 21(2), 303–314.